In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os
PROJECT_DIR = "/content/drive/MyDrive/DL_Group_MGI1"
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
print("Working in:", os.getcwd())


Working in: /content/drive/MyDrive/DL_Group_MGI1


In [15]:
# Install dependencies if not already installed
!pip install -q torch torchvision lightning matplotlib seaborn pathlib scikit-learn scikit-image wandb iterative-stratification torchmetrics


In [5]:
# Run this everytime you update something in the repo!
import os

REPO = "https://github.com/gabrielcastrob/Deep_learning_WUR"

# if project directory is empty clone the repo, otherwise pull the latest changes
if not os.listdir(PROJECT_DIR):
    !git clone {REPO} {PROJECT_DIR}
else:
    !git -C {PROJECT_DIR} pull

%cd {PROJECT_DIR}

print("Working in:", os.getcwd())





remote: Enumerating objects: 20, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 17 (delta 10), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (17/17), 12.68 KiB | 39.00 KiB/s, done.
From https://github.com/gabrielcastrob/Deep_learning_WUR
   505b19f..82318f1  main       -> origin/main
Updating 505b19f..82318f1
Fast-forward
 notebooks/00_SL_CNN.ipynb                          | 110 ++-
 ..._SL_TransferResnet.ipynb => 02_ML_ResNet.ipynb} |   0
 ...{02_multilabel_resnet.ipynb => 03_ML_ViT.ipynb} |   0
 notebooks/03_multilabel_Transformer.ipynb          |   1 -
 notebooks/Level2_ResNet50_UCM_multilabel.ipynb     | 873 +++++++++++++++++++++
 5 files changed, 953 insertions(+), 31 deletions(-)
 rename notebooks/{01_SL_TransferResnet.ipynb => 02_ML_ResNet.ipynb} (100%)
 rename notebooks/{02_multilabel_resnet.ipynb => 03_ML_ViT.ipynb} (100%)
 delete mode 100644 notebooks/03_multilabel_Transformer.ipynb
 crea

Download dataset

In [16]:

"Download the UCMerced Land Use dataset if not already present. "
"The dataset will be saved in the 'ucmdata' directory. "

import os
import zipfile
import subprocess
import shutil
if not os.path.exists('ucmdata'):
    subprocess.run(['git', 'clone', 'https://git.wur.nl/lobry001/ucmdata.git'])
    os.chdir('ucmdata')

    with zipfile.ZipFile('UCMerced_LandUse.zip', 'r') as zip_ref:
        zip_ref.extractall('UCMImages')

    shutil.move('UCMImages/UCMerced_LandUse/Images', '.')
    shutil.rmtree('UCMImages')
    os.remove('README.md')
    os.remove('UCMerced_LandUse.zip')
    print(os.listdir('.'))
    UCM_images_path = "Images/"
    Multilabels_path = "LandUse_Multilabeled.txt"

Load Packages

In [17]:
 import random, shutil, zipfile
from pathlib import Path
import glob

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from torchvision import transforms
import torchvision.models as tvm
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt

import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

import torchmetrics
import re 

L.seed_everything(42, workers=True)
DEVICE = "gpu" if torch.cuda.is_available() else "cpu"
print("Accelerator:", DEVICE)
if DEVICE == "gpu":
    print("GPU:", torch.cuda.get_device_name(0))

INFO: Seed set to 42
INFO:lightning.fabric.utilities.seed:Seed set to 42


Accelerator: cpu


Class list and dataset

In [18]:
class UCMMultilabelDataset(Dataset):
    """
    UCM multilabel land-use dataset.
 
    Folder layout:
        ucmdata/
            Images/
                agricultural/   agricultural00.tif … agricultural99.tif
                airplane/       airplane00.tif      … airplane99.tif
                …               (21 subfolders, already in correct order)
            LandUse_Multilabeled.txt
 
    Label file format (tab-separated):
        IMAGE\LABEL   airplane   bare-soil   buildings   …   water
        agricultural00    0           0            0      …     0
        …
 
    Strategy
    --------
    1. Parse the label file → class names from the header (cols 1…end,
       skipping "IMAGE\LABEL") + label matrix (N × C).
    2. Walk ucmdata/Images/ subfolder-by-subfolder in sorted order, collecting
       every image path into a flat list — same traversal order as the txt file.
    3. Pair image_paths[i]  ↔  label_matrix[i]  by position (no name matching).
    """
 
    def __init__(
        self,
        root_dir: str = "ucmdata",
        label_file: str = "LandUse_Multilabeled.txt",
        transform=None,
        image_ext: str = ".tif",
    ):
        self.root_dir   = root_dir
        self.images_dir = os.path.join(root_dir, "Images")
        self.transform  = transform
        self.image_ext  = image_ext
 
        # ── 1. Parse label file ──────────────────────────────────────────
        label_path = os.path.join(root_dir, label_file)
        self.class_names, self.label_matrix = self._parse_labels(label_path)
        self.num_classes = len(self.class_names)
 
        # ── 2. Collect image paths in sorted subfolder order ─────────────
        self.image_paths = self._collect_image_paths()
 
        # ── 3. Sanity check ──────────────────────────────────────────────
        assert len(self.image_paths) == len(self.label_matrix), (
            f"Mismatch: {len(self.image_paths)} images found but "
            f"{len(self.label_matrix)} label rows in the txt file."
        )
 
    # ------------------------------------------------------------------ #
    def _parse_labels(self, label_path: str):
        """
        Returns:
            class_names  – list[str], length C  (column headers, cols 1…end)
            label_matrix – torch.FloatTensor, shape (N, C)
        """
        with open(label_path, "r") as f:
            lines = [l.strip() for l in f if l.strip()]
 
        # Header row → class names (skip the first column "IMAGE\LABEL")
        header      = lines[0].split("\t")
        class_names = header[1:]          # ['airplane', 'bare-soil', …, 'water']
 
        # Data rows → label matrix (parts[0] is the image name, ignored here)
        rows = []
        for line in lines[1:]:
            parts = line.split("\t")
            label_vals = list(map(int, parts[1:]))
            rows.append(label_vals)
 
        label_matrix = torch.tensor(rows, dtype=torch.float32)  # (N, C)
        return class_names, label_matrix
 
    # ------------------------------------------------------------------ #
    def _collect_image_paths(self) -> list:
        """
        Walks ucmdata/Images/ subfolder-by-subfolder in sorted order.
        Within each subfolder images are also sorted — matching the txt order.
        Returns a flat list of absolute image paths.
        """
        image_paths = []
 
        subfolders = sorted( entry.name for entry in os.scandir(self.images_dir)) 
        
 
        for subfolder in subfolders:
            folder_path = os.path.join(self.images_dir, subfolder)
            files = sorted(
                fname
                for fname in os.listdir(folder_path)
                if fname.lower().endswith(self.image_ext)
            )
            for fname in files:
                image_paths.append(os.path.join(folder_path, fname))
 
        return image_paths
 
    # ------------------------------------------------------------------ #
    def __len__(self) -> int:
        return len(self.image_paths)
 
    def __getitem__(self, idx: int):
        img_path = self.image_paths[idx]
        labels   = self.label_matrix[idx]          # float32 tensor, shape (C,)
 
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
 
        return image, labels
 
    # Utility helpers

    def get_class_names(self) -> list:
        return self.class_names
 
    def get_class_weights(self) -> torch.Tensor:
        """
        Inverse-frequency pos_weight per class for BCEWithLogitsLoss. Combines a Sigmoid layer and the Binary Cross Entropy (BCE) loss into one single class
        Shape: (num_classes,)
        """
        pos = self.label_matrix.sum(dim=0).clamp(min=1)
        neg = (len(self) - self.label_matrix.sum(dim=0)).clamp(min=1)
        return neg / pos
 
    def verify_alignment(self, n_samples: int = 10):
        """
        Prints the first n_samples rows so you can visually cross-check
        image names against the txt file.
        """
        print(f"{'Image path':<55}  {'Active labels'}")
        print("-" * 80)
        for i in range(min(n_samples, len(self))):
            active = [self.class_names[j]
                      for j, v in enumerate(self.label_matrix[i]) if v == 1]
            short = os.path.join(*self.image_paths[i].split(os.sep)[-2:])
            print(f"{short:<55}  {active}")
 

<>:14: SyntaxWarning: invalid escape sequence '\L'
<>:14: SyntaxWarning: invalid escape sequence '\L'
/tmp/ipykernel_31158/2929272064.py:14: SyntaxWarning: invalid escape sequence '\L'
  IMAGE\LABEL   airplane   bare-soil   buildings   …   water


Augmentation  

In [19]:
IMAGE_SIZE = (256, 256)
def get_transforms(split: str = "train"):
    """Standard augmentation for 256x256 UCM aerial tiles."""
    if split == "train":
        return transforms.Compose([
            transforms.Resize(IMAGE_SIZE),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(45),
            transforms.ColorJitter(brightness=0.2, contrast=0.4, saturation=0.3),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], # to prevent exploding gradients 
                                 std=[0.229, 0.224, 0.225]),
        ])
    else:
        return transforms.Compose([
            transforms.Resize(IMAGE_SIZE),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                                 std=[0.229, 0.224, 0.225]),
        ])
 
 

Data split 

In [ ]:
def build_dataloaders(
    root_dir:    str   = "ucmdata",
    label_file:  str   = "LandUse_Multilabeled.txt",
    batch_size:  int   = 32,
    num_workers: int   = 4,
    val_frac:    float = 0.15,
    test_frac:   float = 0.15,
    seed:        int   = 42,
    image_ext:   str   = ".tif",
):
    """
    Returns (train_loader, val_loader, test_loader, class_names, pos_weights).
 
    Usage
    -----
    train_loader, val_loader, test_loader, classes, pos_w = build_dataloaders()
    criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_w)
    """
    
 
    full_ds = UCMMultilabelDataset(
        root_dir=root_dir, label_file=label_file,
        transform=None, image_ext=image_ext,
    )
 
    n = len(full_ds)
    labels_array = full_ds.label_matrix.numpy()  # Convert to numpy
    
    # First split: train+val vs test
    splitter = MultilabelStratifiedShuffleSplit(
        n_splits=1, test_size=test_frac, random_state=seed
    )
    train_val_idx, test_idx = next(splitter.split(np.zeros(n), labels_array))
    
    # Second split: train vs val from train+val
    train_val_labels = labels_array[train_val_idx]
    val_frac_adjusted = val_frac / (1 - test_frac)  # Adjust fraction for remaining data
    
    splitter2 = MultilabelStratifiedShuffleSplit(
        n_splits=1, test_size=val_frac_adjusted, random_state=seed
    )
    train_idx_local, val_idx_local = next(splitter2.split(
        np.zeros(len(train_val_idx)), train_val_labels
    ))
    
    # Map back to original indices
    train_idx = train_val_idx[train_idx_local]
    val_idx = train_val_idx[val_idx_local]
    
    def make_subset(split_name, indices):
        ds = UCMMultilabelDataset(
            root_dir=root_dir, label_file=label_file,
            transform=get_transforms(split_name), image_ext=image_ext,
        )
        return Subset(ds, indices)
    
    train_ds = make_subset("train", train_idx)
    val_ds   = make_subset("val",   val_idx)
    test_ds  = make_subset("test",  test_idx)
    
    kw = dict(batch_size=batch_size, num_workers=num_workers,
              pin_memory=torch.cuda.is_available())
 
    return (
        DataLoader(train_ds, shuffle=True,  **kw),
        DataLoader(val_ds,   shuffle=True, **kw),
        DataLoader(test_ds,  shuffle=False, **kw),
        full_ds.get_class_names(),
        full_ds.get_class_weights(),
    )
 
 

In [21]:
# Full dataloader test
train_loader, val_loader, test_loader, classes, pos_w = build_dataloaders()

# Check class distributions
train_labels = torch.cat([labels for _, labels in train_loader]).numpy()
val_labels = torch.cat([labels for _, labels in val_loader]).numpy()
test_labels = torch.cat([labels for _, labels in test_loader]).numpy()

print("Train class freq:", train_labels.sum(axis=0))
print("Val   class freq:", val_labels.sum(axis=0))
print("Test  class freq:", test_labels.sum(axis=0))

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Train class freq: [ 70. 502. 483. 620.  81.  73.  70.  72. 682.  72. 903. 206.  70.  72.
  70. 706. 142.]
Val   class freq: [ 15. 108. 104. 133.  17.  16.  15.  16. 147.  15. 196.  44.  15.  15.
  15. 152.  31.]
Test  class freq: [ 15. 108. 104. 133.  17.  16.  15.  15. 146.  15. 201.  44.  15.  15.
  15. 151.  30.]


Initialize WANDB

In [22]:

import wandb
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: abelchachoek (abelchachoek-wur) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

Hyperparameters 

In [ ]:
# Define hyperparameters
TODO: ...
run_count = 0


Log WandB

In [ ]:
run = wandb.init(project="multilabel-classification", # or single-label                 
                 name="MODELname_{run_count}", # replace with specific name
                 config={
                     TODO: ..., # link hyperparameters here
                     }) 


Load and train model

Log Results

In [ ]:
run.log({"train_loss": train_loss, "test_loss": test_loss})


run.finish()

Visualize